In [1]:
from _lib import *
from data import *

- load all the historical data and universe

In [2]:
boa = load_star_board()
tickers = boa["ticker"].tolist()
qt.log.info(f"no of tickers on STAR board - [{len(tickers)}]")

[JUSTY.LOG]	2026-05-09 22:26:47,860 - qt.common.help - INFO - no of tickers on STAR board - [609]


- rebalance parameters

In [ ]:
eff_date = dt.date(2026, 3, 14)
annc_date = dt.date(2026, 2, 28)
cutoff_date = dt.date(2026, 1, 31)
hist_start = dt.date(2025, 2, 1)
cutoff_10 = (cutoff_date + pd.offsets.BDay(10)).date()
# sse_holidays = load_sse_holidays()

- get listing date

In [5]:
history = load_historical_data_ohlcv(tickers, hist_start, cutoff_date)

- add scraped data from sse

In [6]:
# merge info_df and tu on ticker
t_ld = get_listing_dates(tickers)

# add listing date to star board
boa = pd.merge(boa, t_ld, on="ticker", how="outer")

- special treatment securities

In [7]:
st_securities = boa[boa['st']]['ticker'].tolist()
qt.log.info(f"[{len(st_securities)}] ST securities: {st_securities}")

[JUSTY.LOG]	2026-05-09 22:27:03,129 - qt.common.help - INFO - [12] ST securities: ['688022', '688033', '688053', '688066', '688076', '688184', '688201', '688270', '688287', '688496', '688622', '688646']


- eligibility
	- Listing time > 6 months. 
		1. If no of securities listed > 12 months is b/w 100 to 150 then requirement changes to > 12 months
	- For securities with daily avg total market_cap since initial listing in top 5, listing time should be > 3 months as of 10th trading days after end date of data (cutoff date)
	- For securities with daily avg total market_cap since initial listing in top 3, listint time should be > 1 month
	- Non-* ST securities
	- No violation of laws/reg, no financial problems etc

In [8]:
history_after_cof = history.copy()

In [12]:
univ = boa[[
	'ticker', 'listing_date', 'name', 'market_cap', 'st', 'shares_total', 
	'shares_tradable'
	]].copy()
univ = univ[~univ['st']].reset_index(drop=True)

univ['month_to_cof'] = univ['listing_date'].apply(lambda d: get_months_from_cutoff(d, cutoff=cutoff_date))
univ['month_to_cof10'] = univ['listing_date'].apply(lambda d: get_months_from_cutoff(d, cutoff=cutoff_10))

# add shares outstanding from yfinance
t_yf = get_yf_info(tickers=univ['ticker'].tolist())
univ = pd.merge(
	univ, 
	t_yf[[
		'ticker', 'shs_os_yf', 
		# 'shs_ff_yf'
		]], 
	on='ticker', how='left')

# if shs_os_yf is not null, use it as shs_total, otherwise use shares_total
# univ.loc[univ['shs_os_yf'] != 0, 'shs_for_avg'] = univ['shs_os_yf']
# univ.loc[univ['shs_os_yf'] == 0, 'shs_for_avg'] = univ['shares_total']
univ['shs_for_avg'] = univ['shares_tradable']

# add avg total mcap and avg value traded
avg_val_traded_n_mcap = get_avg_vol_mcap(history, univ, tickers=univ['ticker'].unique().tolist(), shs_col='shs_for_avg')

univ = pd.merge(univ, avg_val_traded_n_mcap, on='ticker', how='left')
univ['tmcap_rank'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ['vtrad_rank'] = univ['avg_val_traded'].rank(ascending=False, method='min')

In [13]:
if eff_date.month == 6:
	# read weight of existing index
	curr_s50 = load_star50_weights()

	# add weight column
	univ = pd.merge(univ, curr_s50[['ticker', 'curr_weight']], on='ticker', how='left')

elif eff_date.month == 3:
	qt.log.info(f"loading march rebal etf list")
	curr_s50 = load_star50_march_weights()
	univ['curr_weight'] = univ['ticker'].isin(curr_s50)

[JUSTY.LOG]	2026-05-09 22:27:34,905 - qt.common.help - INFO - loading march rebal etf list


- add eligibility

In [14]:
no_of_securities_mt_12m = len(univ[univ['month_to_cof'] >= 12])
listing_month_cutoff = 12 if no_of_securities_mt_12m > 100 else 6
qt.log.info(f"Listing month cutoff: {listing_month_cutoff} months (securities with month_to_cof >= {listing_month_cutoff}: {no_of_securities_mt_12m})")

univ['listing_elig'] = (
	((univ['tmcap_rank'] <= 3) & (univ['month_to_cof'] >= 1)) |
	((univ['tmcap_rank'] <= 5) & (univ['month_to_cof10'] >= 3)) |
	(univ['month_to_cof'] >= listing_month_cutoff)
)

univ = univ[univ['listing_elig']].reset_index(drop=True)
univ['vtrad_rank2'] = univ['avg_val_traded'].rank(ascending=False, method='min')

# drop 10% stocks based on avg value traded rank
univ = univ[univ['vtrad_rank2'] <= 0.9*len(univ)].reset_index(drop=True)
univ['tmcap_rank2'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ = univ.sort_values('tmcap_rank2').reset_index(drop=True)

[JUSTY.LOG]	2026-05-09 22:27:35,576 - qt.common.help - INFO - Listing month cutoff: 12 months (securities with month_to_cof >= 12: 558)


In [15]:
# univ.set_index('ticker')[['shares_total', 'shares_tradable', 'shs_os_yf', 'shs_ff_yf']].plot()

In [17]:
qt.view(univ)

Grid(columns_fit='auto', compress_data=True, css_rules_down=['.number-cell {text-align: left;width: 10;}', '.l…

In [16]:
comp_incl = univ[
	(univ['tmcap_rank2'] <= 40) &
	(univ['curr_weight'].isna())
]
comp_excl = univ[
	(univ['tmcap_rank2'] > 60) &
	(~univ['curr_weight'].isna())
]
print(f"inclusion stocks : \n")
qt.view2(comp_incl)
print(f"exclusion stocks : \n")
qt.view2(comp_excl)

inclusion stocks : 



,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2


exclusion stocks : 



,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2
60,688475,2022-12-28,萤石网络,25.12,False,788.00,788.00,37,37,787.50,788.00,26.40,162.57,245.00,64.00,284.00,False,True,257.00,61.00
61,688425,2021-06-22,铁建重工,25.55,False,"5,333.00","5,333.00",55,55,"5,333.50","5,333.00",25.69,235.90,245.00,65.00,208.00,False,True,182.00,62.00
62,688166,2019-11-08,博瑞医药,28.48,False,427.00,427.00,74,75,423.10,427.00,24.98,666.50,245.00,66.00,62.00,False,True,51.00,63.00
63,688052,2022-04-22,纳芯微,32.69,False,163.00,143.00,45,45,142.53,143.00,24.60,501.99,245.00,67.00,92.00,False,True,76.00,64.00
64,688585,2020-09-28,上纬新材,54.62,False,403.00,403.00,64,64,403.36,403.00,24.58,516.90,245.00,68.00,86.00,False,True,71.00,65.00
65,688676,2021-03-09,XD金盘科,41.94,False,460.00,460.00,58,59,459.78,460.00,24.44,"1,019.68",245.00,69.00,28.00,False,True,21.00,66.00
66,688363,2019-11-12,华熙生物,20.15,False,482.00,482.00,74,75,481.68,482.00,24.40,191.56,245.00,70.00,246.00,False,True,219.00,67.00
67,688037,2019-12-16,芯源微,45.96,False,202.00,202.00,73,73,201.63,202.00,24.09,749.62,245.00,71.00,51.00,False,True,41.00,68.00
68,688343,2023-04-04,云天励飞,31.33,False,360.00,360.00,33,34,359.60,360.00,24.08,"1,036.03",245.00,72.00,27.00,False,True,20.00,69.00
69,688561,2020-07-22,奇安信,21.27,False,682.00,682.00,66,66,682.25,682.00,24.00,333.13,245.00,73.00,146.00,False,True,124.00,70.00


- reporting and stuff

In [ ]:
tmcap_40 = univ[univ['tmcap_rank2'] == 40]['avg_total_mcap'].values[0]
tmcap_60 = univ[univ['tmcap_rank2'] == 60]['avg_total_mcap'].values[0]
qt.log.info(f"TMCap of 40th stock: {tmcap_40:.3f} B CNY, TMCap of 60th stock: {tmcap_60:.3f} B CNY")

- march testing

In [ ]:
univ[univ['ticker'].isin(
	[
		'688498', '688110', '688002',
		'688114', '688278', '688349'	
  	]
)]

In [ ]:
excls = ['688220', '688301', '688385']
incls = ['688213', '688278', '688578']
res_list = ["688608", "688425", "688361", "688568", "688172",]
univ[univ['ticker'].isin(excls)]
univ[univ['ticker'].isin(incls)]
univ[univ['ticker'].isin(res_list)]